In [4]:
from torch.utils.data import Dataset, Subset
import numpy as np
import torch

from torchvision.transforms import v2

augmentations = v2.Compose([
    v2.RandomHorizontalFlip(),
    v2.RandomVerticalFlip(),
    v2.RandomRotation(90.0),  # type: ignore
    v2.ToDtype(torch.float32, scale=True)
])

class DEMTilesDataset(Dataset):
    def __init__(self, dem_path, mask_path, tile_size=64, stride=32, hillshade_path: str | None = None, slope_path: str | None = None, pos_only:bool = False, tile_norm: bool = False, norm_constant: float = 0.0, transforms: bool = False, no_gt: bool = False):
        """
        Simplified dataset for DEM and mask tiling.
        
        Args:
            dem_path: Path to DEM data .npz file
            mask_path: Path to mask .npy file
            tile_size: Size of the square tiles to extract
            stride: Stride for sliding window
        """
        # Load data
        data = np.load(dem_path)
        try:
            self.dem = data["dem"].astype(np.float32)
            self.valid = data["valid"].astype(bool)
        except KeyError:
            try:
                self.dem = data["dataset"].astype(np.float32)
                self.valid = np.ones_like(self.dem, dtype=bool)
            except KeyError as e:
                raise KeyError(f"Required keys not found in DEM file: {e}. Available keys: {list(data.keys())}")
        self.mask = np.load(mask_path).astype(np.uint8) if not no_gt else np.zeros((self.dem.shape[0], self.dem.shape[1]), dtype=np.uint8)
        self.hillshade = np.load(hillshade_path).astype(np.float32) if hillshade_path is not None else None
        self.slope = np.load(slope_path).astype(np.float32) if slope_path is not None else None
        self.tile_size = tile_size
        self.stride = stride
        self.pos_only = pos_only
        self.tile_norm = tile_norm
        self.norm_constant = norm_constant
        self.transforms = augmentations if transforms else None

        print("valid unique:", np.unique(data["valid"]))
        print("dem shape:", data["dem"].shape)
        print("valid shape:", data["valid"].shape)
        print("Mask shape:", self.mask.shape)
        print("Mask sum:", self.mask.sum())

        # Generate all tile coordinates
        H, W = self.dem.shape
        self.coords = []
        
        for y in range(0, H - tile_size + 1, stride):
            for x in range(0, W - tile_size + 1, stride):
                if y + tile_size > H or x + tile_size > W:
                    continue
                # Check if tile has any valid DEM data
                if self.valid[y:y+tile_size, x:x+tile_size].any():
                    if self.pos_only:
                        if self.mask[y:y+tile_size, x:x+tile_size].any():
                            self.coords.append((y, x))
                    else:
                        self.coords.append((y, x))
                #else:
                #    print(f"Skipping tile at ({y}, {x}) - no valid DEM data")

    def __len__(self):
        return len(self.coords)

    def __getitem__(self, idx):
        y, x = self.coords[idx]
        dem_tile = self.dem[y:y+self.tile_size, x:x+self.tile_size]
        mask_tile = self.mask[y:y+self.tile_size, x:x+self.tile_size]
        valid_tile = self.valid[y:y+self.tile_size, x:x+self.tile_size]
        
        # Normalize tile if requested
        if self.tile_norm:
            dem_tile_min = np.min(dem_tile[valid_tile])
            dem_tile = (dem_tile - dem_tile_min) / self.norm_constant
            dem_tile[valid_tile] = np.clip(dem_tile[valid_tile], 0.0, 1.0)
            
        
        # Ensure we always have exactly 3 channels for the model
        if self.hillshade is not None and self.slope is not None:
            hillshade_tile = self.hillshade[y:y+self.tile_size, x:x+self.tile_size]
            slope_tile = self.slope[y:y+self.tile_size, x:x+self.tile_size]
            dem_tile = np.stack((dem_tile, hillshade_tile, slope_tile), axis=0)
        elif self.hillshade is not None:
            hillshade_tile = self.hillshade[y:y+self.tile_size, x:x+self.tile_size]
            dem_tile = np.stack((dem_tile, hillshade_tile, dem_tile), axis=0)
        elif self.slope is not None:
            slope_tile = self.slope[y:y+self.tile_size, x:x+self.tile_size]
            dem_tile = np.stack((dem_tile, slope_tile, dem_tile), axis=0)
        else:
            # Single channel DEM - replicate to 3 channels
            dem_tile = np.stack((dem_tile, dem_tile, dem_tile), axis=0)
        
        # Convert to tensor and prepare for transforms (forced independent memory)
        dem_tensor = torch.from_numpy(np.array(dem_tile, copy=True)).float()
        mask_tensor = torch.from_numpy(np.array(mask_tile, copy=True)).float().unsqueeze(0)
        valid_tensor = torch.from_numpy(np.array(valid_tile, copy=True)).bool().unsqueeze(0)

        # Apply the same transform to both image and mask
        if self.transforms is not None:
            # Stack all tensors to apply the same transform
            stacked = torch.cat([dem_tensor, mask_tensor, valid_tensor.float()], dim=0)
            transformed = self.transforms(stacked)

            # Split back
            dem_tensor = transformed[:3]  # First 3 channels are the image
            mask_tensor = transformed[3:4]  # Next channel is mask
            valid_tensor = transformed[4:].bool()  # Last channel is valid mask

        return {
            'data': dem_tensor.clone(),
            'valid': valid_tensor.squeeze(0).clone(),  # Remove channel dim
            'mask': mask_tensor.squeeze(0).clone(),    # Remove channel dim
            'coords': torch.tensor([y, x], dtype=torch.int32)
        }

In [6]:
import os

DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data', 'processed')
DEM_PATH = os.path.join(DATA_DIR, 'DEM0.80(4)_normalized_train.npz')
MASK_PATH = os.path.join(DATA_DIR, 'mounds_mask0.80(4)_shadowed_train.npy')

dataset = DEMTilesDataset(
    DEM_PATH,
    MASK_PATH,
    128,
    32,
    pos_only=True
)

valid unique: [False  True]
dem shape: (13674, 9791)
valid shape: (13674, 9791)
Mask shape: (13674, 9791)
Mask sum: 379272


In [7]:
import torch
from typing import Any, Dict


def compute_binary_tanimoto_weights_from_dataset(
    dataset,
    mask_key: str = "mask",
    smooth: float = 1e-12,
) -> Dict[str, float]:
    """
    Computes fixed binary class weights from the whole training split.

    Assumptions:
        dataset[i] returns either:
            - (image, mask)
            - {"mask": mask, ...}

        mask shape:
            - [H, W]
            - [1, H, W]

        mask values:
            - binary {0, 1}
            - or values in [0, 1] that can be thresholded externally if needed

    Returns:
        {
            "pos_pixels": ...,
            "neg_pixels": ...,
            "total_pixels": ...,
            "pos_fraction": ...,
            "neg_fraction": ...,
            "inv2_w_pos_raw": ...,
            "inv2_w_neg_raw": ...,
            "inv2_w_pos_norm": ...,
            "inv2_w_neg_norm": ...,
            "inv1_w_pos_raw": ...,
            "inv1_w_neg_raw": ...,
            "inv1_w_pos_norm": ...,
            "inv1_w_neg_norm": ...,
            "pos_to_neg_ratio_inv2": ...,
            "pos_to_neg_ratio_inv1": ...,
        }
    """
    pos_pixels = 0
    total_pixels = 0

    for i in range(len(dataset)):
        sample = dataset[i]

        if isinstance(sample, dict):
            if mask_key not in sample:
                raise KeyError(f"Dataset sample dict does not contain key '{mask_key}'")
            mask = sample[mask_key]
        elif isinstance(sample, (tuple, list)) and len(sample) >= 2:
            mask = sample[1]
        else:
            raise ValueError(
                "Each dataset item must be either (image, mask) or a dict containing mask."
            )

        if not torch.is_tensor(mask):
            mask = torch.as_tensor(mask)

        mask = mask.float()

        if mask.ndim == 3 and mask.shape[0] == 1:
            mask = mask.squeeze(0)
        elif mask.ndim != 2:
            raise ValueError(f"Unsupported mask shape at index {i}: {tuple(mask.shape)}")

        pos_pixels += int(mask.sum().item())
        total_pixels += mask.numel()

    neg_pixels = total_pixels - pos_pixels

    if total_pixels == 0:
        raise ValueError("Training split contains zero pixels.")
    if pos_pixels == 0:
        raise ValueError("Training split contains zero positive pixels.")
    if neg_pixels == 0:
        raise ValueError("Training split contains zero negative pixels.")

    pos_fraction = pos_pixels / total_pixels
    neg_fraction = neg_pixels / total_pixels

    # inverse-square weighting: 1 / V^2
    inv2_w_pos_raw = 1.0 / (pos_pixels ** 2 + smooth)
    inv2_w_neg_raw = 1.0 / (neg_pixels ** 2 + smooth)
    inv2_norm = inv2_w_pos_raw + inv2_w_neg_raw
    inv2_w_pos_norm = inv2_w_pos_raw / inv2_norm
    inv2_w_neg_norm = inv2_w_neg_raw / inv2_norm

    # inverse-volume weighting: 1 / V
    inv1_w_pos_raw = 1.0 / (pos_pixels + smooth)
    inv1_w_neg_raw = 1.0 / (neg_pixels + smooth)
    inv1_norm = inv1_w_pos_raw + inv1_w_neg_raw
    inv1_w_pos_norm = inv1_w_pos_raw / inv1_norm
    inv1_w_neg_norm = inv1_w_neg_raw / inv1_norm

    return {
        "pos_pixels": pos_pixels,
        "neg_pixels": neg_pixels,
        "total_pixels": total_pixels,
        "pos_fraction": pos_fraction,
        "neg_fraction": neg_fraction,

        "inv2_w_pos_raw": inv2_w_pos_raw,
        "inv2_w_neg_raw": inv2_w_neg_raw,
        "inv2_w_pos_norm": inv2_w_pos_norm,
        "inv2_w_neg_norm": inv2_w_neg_norm,
        "pos_to_neg_ratio_inv2": inv2_w_pos_raw / inv2_w_neg_raw,

        "inv1_w_pos_raw": inv1_w_pos_raw,
        "inv1_w_neg_raw": inv1_w_neg_raw,
        "inv1_w_pos_norm": inv1_w_pos_norm,
        "inv1_w_neg_norm": inv1_w_neg_norm,
        "pos_to_neg_ratio_inv1": inv1_w_pos_raw / inv1_w_neg_raw,
    }

In [8]:
# Example usage

stats = compute_binary_tanimoto_weights_from_dataset(dataset)

print("Positive fraction:", stats["pos_fraction"])
print("Negative fraction:", stats["neg_fraction"])

print("Inverse-square normalized weights:")
print("w_pos =", stats["inv2_w_pos_norm"])
print("w_neg =", stats["inv2_w_neg_norm"])

print("Inverse-volume normalized weights:")
print("w_pos =", stats["inv1_w_pos_norm"])
print("w_neg =", stats["inv1_w_neg_norm"])

Positive fraction: 0.016643645069130925
Negative fraction: 0.983356354930869
Inverse-square normalized weights:
w_pos = 0.99971361475371
w_neg = 0.00028638524629000055
Inverse-volume normalized weights:
w_pos = 0.983356354930869
w_neg = 0.016643645069130925
